In [1]:
import os
%pwd

'e:\\Research\\Detecting-Thyroid-Cancer\\research'

In [2]:
os.chdir("../")
%pwd

'e:\\Research\\Detecting-Thyroid-Cancer'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_classes: int

In [4]:
from ThyroidCancer.constants import *
from ThyroidCancer.utils.common import read_yaml, create_directories
import tensorflow as tf


In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Thyroid Data")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
            params_classes=params.CLASSES
        )

        return training_config
    

In [6]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        # Rebuild exact same architecture (EfficientNetB0)
        base = tf.keras.applications.EfficientNetB0(
            input_shape=self.config.params_image_size,
            weights='imagenet',
            include_top=False
        )
        
        x = tf.keras.layers.GlobalAveragePooling2D(name="avg_pool")(base.output)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.5, name="top_dropout")(x)
        predictions = tf.keras.layers.Dense(
            units=self.config.params_classes,
            activation="softmax",
            name="predictions"
        )(x)

        self.model = tf.keras.models.Model(inputs=base.input, outputs=predictions)

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        # Now load the weights
        self.model.load_weights(self.config.updated_base_model_path)
        print(f"Model rebuilt, compiled, and weights loaded from:\n{self.config.updated_base_model_path}")

    def train_valid_generator(self):
        datagenerator_kwargs = dict(rescale=1./255, validation_split=0.20)
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_gen = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)
        self.valid_generator = valid_gen.flow_from_directory(
            self.config.training_data, subset="validation", shuffle=False, **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40, horizontal_flip=True, width_shift_range=0.2,
                height_shift_range=0.2, shear_range=0.2, zoom_range=0.2, **datagenerator_kwargs
            )
        else:
            train_gen = valid_gen

        self.train_generator = train_gen.flow_from_directory(
            self.config.training_data, subset="training", shuffle=True, **dataflow_kwargs
        )

    @staticmethod
    def save_trained_model(path: Path, model: tf.keras.Model):
        path = path.with_suffix(".h5")
        model.save_weights(path)
        print(f"Trained weights saved: {path}")

    def train(self):
        steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        val_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=steps_per_epoch,
            validation_steps=val_steps,
            validation_data=self.valid_generator
        )
        self.save_trained_model(self.config.trained_model_path, self.model)

In [7]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2025-11-20 20:21:03,888: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-11-20 20:21:03,899: INFO: common: yaml file: params.yaml loaded successfully]
[2025-11-20 20:21:03,901: INFO: common: created directory at: artifacts]
[2025-11-20 20:21:03,903: INFO: common: created directory at: artifacts\training]
Model rebuilt, compiled, and weights loaded from:
artifacts\prepare_base_model\base_model_updated.h5
Found 623 images belonging to 2 classes.
Found 2492 images belonging to 2 classes.
155/155 [==============================] - 396s 2s/step - loss: 1.1635 - accuracy: 0.5808 - val_loss: 1.1435 - val_accuracy: 0.4539
Trained weights saved: artifacts\training\model.h5
